# Dokumentasi Komputasi Simulasi Keselamatan Pengereman Kendaraan
**Mata Kuliah:** Sains Komputasi / Fisika Komputasi  
**Program Studi:** Magister Fisika, Universitas Padjadjaran  
**Topik Kajian:** Pemodelan Dinamika Pengereman GLBB, Diskritisasi Euler Orde Satu, Analisis Asal-Usul Konstanta Gravitasi pada Gerak Horizontal, Penerapan Syarat Batas, dan Diagram Alir Program Menyeluruh

---

## Ringkasan Eksekutif
Dokumen ini menyajikan kajian matematis, analitis, dan algoritmik dari sistem komputasi pengereman darurat kendaraan menghadapi bahaya barikade jalan rusak (*potholes*). Tiga fokus utama yang dikaji secara mendalam meliputi:
1. **Asal-Usul Konstanta Gravitasi ($g$) pada Gerak Horizontal & Formulasi Euler Orde Satu:** Pembuktian fisis mengapa percepatan gravitasi $g$ mutlak menentukan perlambatan horizontal $a = \mu g$ melalui gaya normal ban-jalan, serta formulasi beda maju (*forward difference*) dan skema semi-implisit *Euler-Cromer*.
2. **Penerapan Syarat Batas Fisis (*Boundary Conditions*):** Formulasi syarat awal ($t=0$), kendala gesek satu arah ($v \ge 0$), batas spasial barikade perbaikan jalan ($x = d$), serta mitigasi impuls-momentum kantung udara (*airbag*).
3. **Diagram Alir Program Menyeluruh (*Complete Program Flowchart*):** Pemetaan grafis Mermaid yang merepresentasikan seluruh siklus hidup program: inisialisasi lingkungan 3D perkotaan, *toggle* mode siang/malam, loop integrasi waktu-nyata 60 FPS, pemicu benturan, hingga pencatatan tabel hasil percobaan.


---

## 1. Ekspresi Persamaan Diferensial & Analisis Konstanta Gravitasi

### 1.1 Mengapa Konstanta Gravitasi ($g$) Muncul pada Gerak Horizontal?
Sering kali timbul pertanyaan konseptual: **"Kendaraan bergerak murni horizontal sepanjang sumbu-$x$, mengapa percepatan gravitasi bumi ($g$) muncul di dalam persamaan gerak pengeremannya?"**

Jawaban fisisnya terletak pada **kopling gaya vertikal dan horizontal melalui fenomena gesekan kering (Hukum Coulomb)**.

#### A. Keseimbangan Dinamika pada Sumbu Vertikal ($y$)
Tinjau kendaraan bermassa $m$ yang melaju di atas jalan mendatar. Pada arah vertikal (tegak lurus permukaan jalan), kendaraan tidak mengalami pergerakan maupun percepatan ($\sum F_y = 0$).

Gaya-gaya yang bekerja pada sumbu vertikal adalah:
1. **Gaya Berat ($W$):** Tarikan gravitasi bumi yang menarik massa kendaraan ke bawah:
   $$W = mg$$
2. **Gaya Normal ($N$):** Gaya kontak elastis dari permukaan jalan yang menahan kendaraan ke atas agar tidak amblas ke dalam tanah.

Berdasarkan **Hukum I Newton** pada sumbu-$y$:
$$\sum F_y = 0 \implies N - W = 0 \implies N = mg$$

Gaya normal $N$ inilah yang menjadi ukuran seberapa kuat ban ditekan ke aspal oleh gravitasi bumi.

#### B. Asal-Usul Gaya Pengereman pada Sumbu Horizontal ($x$)
Pada kendaraan darat, satu-satunya gaya eksternal horizontal yang mampu memperlambat laju kendaraan saat rem diinjak adalah **gaya gesek kinetik ($f_k$)** antara tapak ban yang selip dan permukaan jalan.

Menurut Hukum Empiris Gesekan Kering Amontons-Coulomb, besar gaya gesek kinetik sebanding secara linier dengan gaya normal yang menekan kedua permukaan:
$$f_k = \mu N$$
dengan $\mu$ adalah koefisien gesek kinetik permukaan jalan.

Substitusi nilai gaya normal $N = mg$ ke dalam persamaan gesekan:
$$f_k = \mu (mg) = \mu mg$$

#### C. Hukum II Newton pada Sumbu Gerak Horizontal ($x$)
Gaya gesek kinetik selalu berarah melawan arah gerak vektor kecepatan ($\vec{v}$). Berdasarkan Hukum II Newton pada sumbu horizontal:
$$\sum F_x = m a_x \implies -f_k = m a_x$$
$$-(\mu mg) = m a_x$$

Perhatikan bahwa **massa kendaraan $m$ muncul di kedua ruas persamaan dan saling meniadakan (*cancels out*)**:
$$a_x = -\mu g \implies a_{\text{brake}} = |a_x| = \mu g$$

#### D. Kesimpulan Fisis
1. **Gravitasi sebagai Pemicu Gaya Gesek:** Meskipun lintasan mobil horizontal, gaya gesek horizontal **tidak akan pernah ada tanpa adanya gaya berat vertikal**. Gravitasi bumi $g$ adalah agen yang menekan roda ke aspal sehingga tercipta gaya normal $N$.
2. **Kondisi Nirgravitasi ($g \to 0$):** Jika simulasi ini dijalankan di luar angkasa atau kondisi tanpa gravitasi ($g = 0$), maka $N = 0 \implies f_k = 0 \implies a = 0$. Akibatnya, sekeras apa pun pengemudi menginjak rem, mobil tidak akan pernah melambat dan akan terus meluncur lurus selamanya dengan laju tetap $v_0$ (Hukum I Newton).
3. **Gravitasi di Planet Lain:** Di Bulan ($g_{\text{bulan}} \approx 1{,}62\,\text{m/s}^2$), gaya cengkeram ban turun drastis hingga $\approx 1/6$ nilai di Bumi. Jarak pengereman yang dibutuhkan kendaraan di Bulan membengkak $6$ kali lipat lebih jauh untuk kecepatan awal yang sama.

---

### 1.2 Sistem Persamaan Diferensial Terkopel
Dinamika pengereman kendaraan 1D dinyatakan dalam pasangan **Persamaan Diferensial Biasa (PDB) Terkopel Orde Satu**:
$$\begin{cases}
\dfrac{\mathrm{d}v(t)}{\mathrm{d}t} = -a_{\text{brake}} = -\mu g \\[8pt]
\dfrac{\mathrm{d}x(t)}{\mathrm{d}t} = v(t)
\end{cases}$$

### 1.3 Solusi Analitik Eksak (GLBB)
Integrasi langsung terhadap waktu dengan syarat awal $x(0) = 0$ dan $v(0) = v_0$ memberikan:
$$v(t) = v_0 - \mu g t$$
$$x(t) = v_0 t - \frac{1}{2}\mu g t^2$$

Kendaraan berhenti ketika $v(t_{\text{henti}}) = 0$:
$$t_{\text{henti}} = \frac{v_0}{\mu g}$$
$$s_{\text{analitik}} = x(t_{\text{henti}}) = \frac{v_0^2}{2\mu g}$$

---

### 1.4 Diskritisasi Numerik Metode Euler Orde Satu
Dalam komputasi numerik, kontinuum waktu $t$ dipecah menjadi kisi waktu seragam berjarak $\Delta t$:
$$t_n = n \Delta t, \quad n = 0, 1, 2, \dots$$

Melalui ekspansi deret Taylor orde satu:
$$v(t_n + \Delta t) = v(t_n) + \left.\frac{\mathrm{d}v}{\mathrm{d}t}\right|_{t_n} \Delta t + \mathcal{O}(\Delta t^2) \implies v_{n+1} = v_n - \mu g \Delta t$$
$$x(t_n + \Delta t) = x(t_n) + \left.\frac{\mathrm{d}x}{\mathrm{d}t}\right|_{t_n} \Delta t + \mathcal{O}(\Delta t^2) \implies x_{n+1} = x_n + v_n \Delta t$$

#### Skema Semi-Implisit (Euler-Cromer) pada Program
Pada simulator WebGL (`project.js`), algoritma mengimplementasikan varian **Euler-Cromer (Symplectic Euler)** di mana kecepatan baru $v_{n+1}$ dihitung terlebih dahulu, kemudian langsung digunakan untuk memperbarui posisi $x_{n+1}$:
$$\begin{aligned}
v_{n+1} &= \max\left(0,\, v_n - \mu g \Delta t\right) \\
x_{n+1} &= x_n + v_{n+1} \Delta t
\end{aligned}$$

Keunggulan Euler-Cromer pada simulasi pengereman:
- **Konservasi Energi & Kestabilan:** Menghilangkan artifisial *energy growth* yang umum terjadi pada Forward Euler standar.
- **Pencegahan Overshoot Posisi:** Penjepitan $\max(0, \dots)$ menjamin mobil tidak mundur saat langkah integrasi terakhir mendekati kecepatan nol.


---

## 2. Penerapan Syarat Batas (*Boundary Conditions*)

Penyelesaian persamaan gerak pada simulasi keselamatan pengereman kendaraan melibatkan beberapa syarat batas fisis, geometris, dan mekanika tumbukan:

### 2.1 Syarat Awal (*Initial Conditions* pada $t = 0$)
Sesaat sebelum proses pengereman dimulai:
$$\begin{cases}
x(0) = 0\,\text{m} & (\text{titik awal acuan pengereman}) \\
v(0) = v_0 = \dfrac{v_{0,\,\text{km/h}}}{3{,}6}\,\text{m/s} & (\text{kecepatan translasi awal}) \\
t_{\text{gerak}} = 0\,\text{s} & (\text{timer penghitung durasi pengereman direset}) \\
\text{status} = \text{"running"}
\end{cases}$$

### 2.2 Syarat Batas Gesekan Satu Arah (*Unilateral Constraint* $v \ge 0$)
Gaya gesek bersifat disipatif pasif. Gaya gesek kinetik hanya bekerja melawan arah laju selama benda bergerak ($v > 0$). Ketika mobil sudah berhenti diam ($v = 0$), gaya gesek kinetik lenyap ($f_k = 0$).

Jika tidak dibatasi, persamaan $\frac{\mathrm{d}v}{\mathrm{d}t} = -\mu g$ akan menyebabkan kecepatan bernilai negatif ($v < 0$), yang secara fisis salah karena berarti gaya gesek mendorong mobil melaju mundur!

Secara komputasi diterapkan operator pemangkasan (*clamping*):
$$v_{n+1} = \max(0,\, v_n - \mu g \Delta t)$$

### 2.3 Syarat Batas Kriteria Berhenti Selamat ($s \le d$)
Jika jarak henti teoritis lebih kecil atau sama dengan jarak ke barikade ($s \le d$):
$$\text{Kondisi:} \quad v_{n+1} \le 0{,}01\,\text{m/s} \quad \text{pada saat } x < d$$
Aksi komputasi:
$$\begin{cases}
v_{\text{akhir}} = 0\,\text{m/s} \\
x_{\text{akhir}} = x_{n+1} \le d \\
\Delta x = d - x_{\text{akhir}} \ge 0 \quad (\text{sisa margin jarak aman}) \\
\text{simState} = \text{"safe"}
\end{cases}$$
Kendaraan berhasil berhenti aman sebelum menyentuh barikade, sehingga terhindar dari bahaya jatuh ke zona jalan berlubang.

### 2.4 Syarat Batas Benturan Rintangan ($s > d$)
Jika jarak henti melebihi jarak rintangan ($s > d$), kendaraan tidak memiliki ruang henti yang cukup dan menabrak barikade perbaikan jalan di $x = d$.

Syarat batas batas spasial Dirichlet:
$$x(t) \le d, \quad \forall t$$
Saat ujung depan kendaraan mencapai koordinat barikade:
$$\text{Kondisi:} \quad x_{n+1} \ge d \quad \text{saat } v_{n+1} > 0$$
Aksi komputasi:
$$\begin{cases}
x_{\text{kontak}} = d \\
v_{\text{benturan}} = v_{n+1} \approx \sqrt{v_0^2 - 2\mu g d} \\
\text{simState} = \text{"crashed"}
\end{cases}$$
Kendaraan mengalami benturan langsung dengan barikade konstruksi, memicu getaran kamera (*camera shake*), kemiringan barikade, dan pengembangan kantung udara (*airbag*).

### 2.5 Syarat Batas Deformasi & Impuls Airbag
Pada saat benturan, momentum linier kendaraan diserap hingga berhenti dalam selang waktu redam $\Delta t$:
$$\Delta p = m v_{\text{benturan}}$$
$$I = \int_0^{\Delta t} F(t)\,\mathrm{d}t = \Delta p$$

- **Tanpa Airbag (Benturan Kaku):** Waktu deselerasi tubuh sangat singkat ($\Delta t_{\text{kaku}} \approx 0{,}015\,\text{s}$). Gaya rata-rata yang dialami organ vital:
  $$\bar{F}_{\text{kaku}} = \frac{m v_{\text{benturan}}}{\Delta t_{\text{kaku}}} \quad (\text{sangat fatal, mencapai ribuan kN})$$
- **Dengan Airbag:** Kantung gas mengembang dan mengempis terkontrol, memperpanjang durasi kontak menjadi $\Delta t_{\text{airbag}} \approx 0{,}10 - 0{,}20\,\text{s}$.
  Dengan profil gaya impuls sinusoidal $F(t) = F_{\text{peak}} \sin\left(\frac{\pi t}{\Delta t_{\text{airbag}}}\right)$, gaya puncak tereduksi:
  $$F_{\text{peak}} = \frac{\pi}{2} \left(\frac{m v_{\text{benturan}}}{\Delta t_{\text{airbag}}}\right)$$
  Durasi yang $7\times$ lebih panjang mereduksi gaya puncak sebesar **$\approx 85\%$**, secara efektif menyelamatkan pengemudi dari cedera fatal.


---

## 3. Diagram Alir Program (*Program Flowchart*)

### 3.1 Visualisasi Diagram Alir
Berikut adalah diagram alir algoritma numerik simulasi pengereman kendaraan berbasis fisika komputasi:

<div align="center">
  <img src="flowchart.svg" alt="Diagram Alir Simulasi Pengereman Kendaraan" width="100%" style="max-width:760px; border-radius:8px; border:1px solid #e2e8f0; margin: 12px 0;"/>
</div>

---

### 3.2 Diagram Alir Mermaid
Struktur logika program dinyatakan dalam diagram alir Mermaid ringkas dan terfokus berikut:

```mermaid
flowchart TD
    Start(["Mulai"]) --> Input[/"Input Parameter:<br/>v₀, m, μ, d, Δt_airbag"/]
    Input --> Init["Kondisi Awal:<br/>t = 0, x = 0, v = v₀, a = -μ·g"]
    Init --> Euler["Integrasi Euler-Cromer (dt):<br/>vₙ₊₁ = max(0, vₙ - μ·g·dt)<br/>xₙ₊₁ = xₙ + vₙ₊₁·dt"]
    
    Euler --> CheckCrash{"xₙ₊₁ ≥ d ?<br/>(Tabrak Barikade)"}
    
    CheckCrash -- "Ya (s > d)" --> Crash["Kondisi Tabrakan:<br/>v_impak = vₙ₊₁<br/>F_peak = (π/2) · (m · v_impak / Δt_airbag)"]
    
    CheckCrash -- "Tidak" --> CheckStop{"vₙ₊₁ == 0 ?<br/>(Berhenti)"}
    
    CheckStop -- "Ya (s ≤ d)" --> Safe["Berhenti Selamat:<br/>v = 0<br/>Margin Aman: Δx = d - xₙ₊₁"]
    
    CheckStop -- "Tidak (Lanjut Loop)" --> Euler
    
    Crash --> Done(["Selesai / Catat Data"])
    Safe --> Done
```

---

### 3.3 Tahapan Inti Algoritma
1. **Input & Konversi Satuan:** Menerima kecepatan awal $v_0$ (dikonversi ke $\text{m/s}$), massa kendaraan $m$, koefisien gesek jalan $\mu$, jarak barikade $d$, dan waktu redam kantung udara $\Delta t_{\text{airbag}}$.
2. **Inisialisasi Keadaan Awal:** Menetapkan $t = 0\,\text{s}$, $x_0 = 0\,\text{m}$, dan deselerasi gesekan $a = -\mu g$.
3. **Loop Integrasi Waktu (Euler-Cromer per $\Delta t$):**
   - Menghitung kecepatan baru dengan syarat batas $v \ge 0$:
     $$v_{n+1} = \max\left(0,\, v_n - \mu g \Delta t\right)$$
   - Memperbarui posisi linier:
     $$x_{n+1} = x_n + v_{n+1} \Delta t$$
4. **Pemeriksaan Syarat Batas Berlapis:**
   - **Batas Rintangan ($x_{n+1} \ge d$):** Jika posisi mencapai barikade sebelum laju habis ($s > d$), tabrakan terjadi. Posisi dijepit di $x = d$ dan gaya puncak impuls dihitung:
     $$F_{\text{peak}} = \frac{\pi}{2} \left(\frac{m v_{\text{impak}}}{\Delta t_{\text{airbag}}}\right)$$
   - **Batas Berhenti Aman ($v_{n+1} = 0$):** Jika laju habis sebelum menyentuh barikade ($s \le d$), kendaraan berhenti selamat dengan margin keselamatan:
     $$\Delta x = d - x_{n+1}$$


---

## 4. Tabel Sintesis & Kesimpulan Analitis

### 4.1 Tabel Komparasi Metode Analitik vs Euler Orde Satu
| Parameter Fisis / Aspek | Solusi Analitik GLBB | Diskritisasi Euler-Cromer Orde Satu |
| :--- | :--- | :--- |
| **Persamaan Kecepatan** | $v(t) = v_0 - \mu g t$ | $v_{n+1} = \max(0,\, v_n - \mu g \Delta t)$ |
| **Persamaan Posisi** | $x(t) = v_0 t - \frac{1}{2}\mu g t^2$ | $x_{n+1} = x_n + v_{n+1}\Delta t$ |
| **Peran Gravitasi ($g$)**| Penentu gaya normal $N = mg$ | Penentu akselerasi diskrit $a = \mu g$ |
| **Galat Pemotongan Lokal (LTE)** | Eksak ($0$) | $\mathcal{O}(\Delta t^2)$ |
| **Galat Pemotongan Global (GTE)**| Eksak ($0$) | $\mathcal{O}(\Delta t)$ |
| **Komputasi Syarat Batas** | Aljabar langsung ($s \le d$) | Pemantauan interaktif waktu-nyata per $\Delta t$ |
| **Generalisasi Sistem** | Sulit jika $\mu(v, x)$ tidak konstan | Sangat mudah untuk gesekan dinamis/non-linier |

### 4.2 Matriks Penerapan Syarat Batas
| Syarat Batas | Formulasi Matematis | Aksi Komputasi dalam Program |
| :--- | :--- | :--- |
| **Syarat Awal ($t = 0$)** | $x(0) = 0,\; v(0) = v_0,\; t = 0$ | Reset register kinematika, timer gerak, dan partikel |
| **Batas Gesek Kinetik** | $v(t) \ge 0 \quad (\text{unilateral})$ | Pemangkasan $\max(0, v_{n+1})$ agar mobil tidak meluncur mundur |
| **Batas Berhenti Aman** | $s \le d \iff \frac{v_0^2}{2\mu g} \le d$ | Status `"safe"`, hitung sisa jarak $\Delta x = d - s$, hindari lubang |
| **Batas Barikade Konstruksi**| $x \ge d$ saat $v > 0$ | Jepit $x = d$, catat $v_{\text{benturan}}$, alihkan ke modul impuls |
| **Batas Deformasi Airbag** | $I = \int_0^{\Delta t} F(t)\,\mathrm{d}t = m v_{\text{benturan}}$ | Perpanjang durasi $\Delta t$, reduksi gaya puncak $F_{\text{peak}} = \frac{\pi m v}{2 \Delta t}$ |

### 4.3 Kesimpulan Akhir
1. **Peran Vital Gravitasi pada Gerak Horizontal:** Meskipun gerak translasi pengereman berlangsung horizontal, gaya gesek penghenti $f_k = \mu mg$ sepenuhnya bergantung pada tarikan gravitasi bumi yang menghasilkan gaya normal vertikal $N = mg$. Tanpa gravitasi ($g = 0$), mobil tidak dapat mengerem sama sekali.
2. **Kekokohan Euler-Cromer:** Skema semi-implisit Euler-Cromer dengan pembatasan langkah waktu $\Delta t \le 0{,}033\,\text{s}$ mampu meniru solusi analitik GLBB dengan galat posisi sangat kecil ($< 0{,}05\,\text{m}$), sekaligus mencegah anomali *overshoot* atau pergerakan mundur saat laju mobil habis.
3. **Kontekstualisasi Keselamatan:** Integrasi objek barikade peringatan perbaikan jalan dan zona jalan berlubang memberikan alasan fisis dan rekayasa yang kuat terhadap pentingnya kepatuhan batas kecepatan dan efektivitas sistem keselamatan kantung udara (*airbag*).
